In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import json
import os
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

from hedgementation_utils.io.io_manager import HedgementationIOManager

import copy
from typing import Any
from hedgementation_utils.visualization.visualization_utils import show_satellite_img, show_hedge_raster
from src.visualization.visualization_utils import get_and_display_prediction


TRAIN_FOLDS = json.loads(os.environ["TRAIN_FOLDS"] if os.environ["TRAIN_FOLDS"] else [0,1,2]) 
VALID_FOLDS = json.loads(os.environ["VALID_FOLDS"] if os.environ["VALID_FOLDS"] else [3])
TEST_FOLDS = json.loads(os.environ["TEST_FOLDS"] if os.environ["TEST_FOLDS"] else [4])

DATASET_ROOT = os.environ["DATASET_ROOT"]
RPG_MASK_SUBDIR = os.environ["RPG_MASK_SUBDIR"]

In [2]:
#metadata = gpd.read_file(os.path.join(DATASET_ROOT, "metadata.geojson")).to_crs("EPSG:3857")
root = "/scratch/nathan/data/hedgementation_1.3_test/"
metadata = gpd.read_file(f"{root}/metadata.geojson").to_crs("EPSG:3857")
metadata = metadata[:20]
metadata.to_file(f"{root}/metadata.geojson")

In [ ]:
import rasterio
import re
from datetime import datetime
from datetime import timedelta

def band_names_to_dates(band_names):
    band_names = list(band_names)
    return sorted([
        fix_band_name(bn) for bn in band_names
    ])[::10]



def fix_band_name(band_name):
    match = re.match(r'(\d{4})-(\d{2})-(\d+)(_.*)', band_name)
    if not match:
        return band_name  
    
    week_year, month, day_of_year, _ = match.groups()
    week_year = int(week_year)
    month = int(month)
    day_of_year = int(day_of_year)
    
    base_date = datetime(week_year, 1, 1)
    
    actual_date = base_date + timedelta(days=day_of_year - 1)
    
    correct_date = actual_date.strftime('%Y-%m-%d')
    
    return correct_date.replace("-","")


def get_dates_S2(ind, dataset_root=DATASET_ROOT, X_tif_dir="X"):
    manager = HedgementationIOManager(dataset_root=dataset_root, X_dir=X_tif_dir)

    _, dates_series = manager.load_X(ind, file_type="tif", return_dates=True)

    dates_s2 = dates_series.str.replace("-","").values 

    return {str(i):d for i,d in enumerate(dates_s2)}

In [4]:
get_dates_S2(0, dataset_root=root)

{'0': '20210918',
 '1': '20210921',
 '2': '20210923',
 '3': '20210926',
 '4': '20210928',
 '5': '20211001',
 '6': '20211003',
 '7': '20211006',
 '8': '20211008',
 '9': '20211011',
 '10': '20211013',
 '11': '20211016',
 '12': '20211018',
 '13': '20211021',
 '14': '20211023',
 '15': '20211026',
 '16': '20211028',
 '17': '20211031',
 '18': '20211102',
 '19': '20211105',
 '20': '20211107',
 '21': '20211110',
 '22': '20211112',
 '23': '20211115',
 '24': '20211117',
 '25': '20211120',
 '26': '20211122',
 '27': '20211125',
 '28': '20211127',
 '29': '20211130',
 '30': '20211202',
 '31': '20211205',
 '32': '20211207',
 '33': '20211210',
 '34': '20211212',
 '35': '20211215',
 '36': '20211217',
 '37': '20211220',
 '38': '20211222',
 '39': '20211225',
 '40': '20211227',
 '41': '20211230',
 '42': '20220101',
 '43': '20220104',
 '44': '20220106',
 '45': '20220109',
 '46': '20220111',
 '47': '20220114',
 '48': '20220116',
 '49': '20220119',
 '50': '20220121',
 '51': '20220124',
 '52': '20220126',
 '5

In [5]:
get_dates_S2(
        ind=0,
        X_tif_pattern="X/X_{}.tif",
        dataset_root=root
    )

{'0': '20210918',
 '1': '20210921',
 '2': '20210923',
 '3': '20210926',
 '4': '20210928',
 '5': '20211001',
 '6': '20211003',
 '7': '20211006',
 '8': '20211008',
 '9': '20211011',
 '10': '20211013',
 '11': '20211016',
 '12': '20211018',
 '13': '20211021',
 '14': '20211023',
 '15': '20211026',
 '16': '20211028',
 '17': '20211031',
 '18': '20211102',
 '19': '20211105',
 '20': '20211107',
 '21': '20211110',
 '22': '20211112',
 '23': '20211115',
 '24': '20211117',
 '25': '20211120',
 '26': '20211122',
 '27': '20211125',
 '28': '20211127',
 '29': '20211130',
 '30': '20211202',
 '31': '20211205',
 '32': '20211207',
 '33': '20211210',
 '34': '20211212',
 '35': '20211215',
 '36': '20211217',
 '37': '20211220',
 '38': '20211222',
 '39': '20211225',
 '40': '20211227',
 '41': '20211230',
 '42': '20220101',
 '43': '20220104',
 '44': '20220106',
 '45': '20220109',
 '46': '20220111',
 '47': '20220114',
 '48': '20220116',
 '49': '20220119',
 '50': '20220121',
 '51': '20220124',
 '52': '20220126',
 '5

In [6]:
metadata["dates-S2"] = metadata["ID_PATCH"].apply(
    lambda x: get_dates_S2(
        ind=x,
        X_tif_pattern="X/X_{}.tif",
        dataset_root=root
    )
)
metadata.to_file(f"{root}/metadata.geojson")

In [7]:
from src.training.dataset_dataloader import HedgementationDataset


dataset = HedgementationDataset(
    metadata_frame=metadata,
    root_dir="/scratch/nathan/data/hedgementation_1.3_test",
    load_X_cloud=True,
    load_y_id=True,
    reference_date="2021-09-17",
    cloud_threshold=0.2
)

(X, dates, X_cloud), y, masks = dataset.__getitem__(4)
X.shape, dates.shape, X_cloud.shape

(torch.Size([63, 10, 128, 128]),
 torch.Size([63]),
 torch.Size([63, 2, 128, 128]))